In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from tabulate import tabulate
import warnings

# Abaikan warning xarray/numpy yang tidak krusial
warnings.filterwarnings("ignore")

# --- 1. KONFIGURASI DIREKTORI ---
LOCAL_RUN = True  

if LOCAL_RUN:
    base_path = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\1-seleksi_model\wget_script\analisis"
    output_path = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\1-seleksi_model\output"
else:
    base_path = os.path.join("data", "1-seleksi_model", "wget_script", "analisis")
    output_path = os.path.join("data", "1-seleksi_model", "output")

era5_path = os.path.join(base_path, "ERA5")

if not os.path.exists(output_path):
    os.makedirs(output_path)

models = {
    "CanESM5": "canesm", "EC-Earth3": "ecearth", "IPSL-CM6A-LR": "ipsl",
    "MPI-ESM1-2-HR": "mpi-hr", "MPI-ESM1-2-LR": "mpi-lr",
    "NorESM2-LM": "noresm-lm", "NorESM2-MM": "noresm-mm"
}
variables = ["sfcWind", "rsds"]

# PERUBAHAN: CanESM5 diganti jadi Hexagon ('h') biar ga mirip ERA5
model_styles = {
    "CanESM5":      {"m": "h", "c": "black"},
    "EC-Earth3":    {"m": "s", "c": "dimgray"},
    "IPSL-CM6A-LR": {"m": "^", "c": "gray"},
    "MPI-ESM1-2-HR":{"m": "D", "c": "darkgray"},
    "MPI-ESM1-2-LR":{"m": "v", "c": "silver"},
    "NorESM2-LM":   {"m": "p", "c": "lightgray"},
    "NorESM2-MM":   {"m": "*", "c": "black"}
}

def get_stats(obs_data, mod_data):
    obs_adj, mod_adj = xr.align(obs_data, mod_data, join='inner')
    mod_adj = mod_adj.interp_like(obs_adj, method='nearest')
    o_flat, m_flat = obs_adj.values.flatten(), mod_adj.values.flatten()
    
    mask = ~np.isnan(o_flat) & ~np.isnan(m_flat)
    o_flat, m_flat = o_flat[mask], m_flat[mask]
    
    if len(o_flat) == 0: return None, None, None
    r = np.corrcoef(o_flat, m_flat)[0, 1]
    std_norm = np.std(m_flat) / np.std(o_flat)
    rmsd_norm = np.sqrt(1 + std_norm**2 - 2 * std_norm * r)
    return r, std_norm, rmsd_norm

# --- 2. PROSES UTAMA (MULTIPANEL COMPACT & PRESISI) ---
fig, axes = plt.subplots(2, 2, figsize=(20, 18), subplot_kw={'projection': 'polar'})
all_stats = []

panel_labels = [['A', 'B'], ['C', 'D']]

for row_idx, var in enumerate(variables):
    print(f"\n{'='*65}\nMEMPROSES VARIABEL: {var}\n{'='*65}")
    
    for col_idx, mode in enumerate(["monthly", "climatology"]):
        ax = axes[row_idx, col_idx]
        ax.set_thetamin(0)
        ax.set_thetamax(90)
        
        # A. GARIS BANTU RMSD
        for rmsd in [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]:
            circle = plt.Circle((1, 0), rmsd, transform=ax.transData._b, 
                                color='black', linestyle='--', alpha=0.4, fill=False, linewidth=1.8)
            ax.add_artist(circle)
            
        # B. GARIS REFERENSI SD=1
        t_ref = np.linspace(0, np.pi/2, 100)
        ax.plot(t_ref, np.ones_like(t_ref), color='black', linewidth=2.5, alpha=0.85, zorder=5)

        # C. REFERENSI ERA5 (Lingkaran Hitam Murni)
        ax.plot(0, 1, 'ko', markersize=15, label='Ref (ERA5)', zorder=20)

        # D. PLOT DATA MODEL
        for m_name, m_alias in models.items():
            try:
                model_file = os.path.join(base_path, f"{var}_{m_name}_1991-2020_Indo.nc")
                obs_file = os.path.join(era5_path, f"{var}_era5-daily_{m_alias}-grid_1991-2020.nc")
                
                if not os.path.exists(model_file) or not os.path.exists(obs_file):
                    continue

                ds_m = xr.open_dataset(model_file)[var]
                ds_o_raw = xr.open_dataset(obs_file)
                
                t_d = 'valid_time' if 'valid_time' in ds_o_raw.dims else 'time'
                ds_o = ds_o_raw[var].rename({t_d: 'time'})

                if mode == "monthly":
                    ds_m = ds_m.resample(time='1MS').mean()
                    ds_o = ds_o.resample(time='1MS').mean()
                    ds_m = ds_m.assign_coords(time=ds_m.time.dt.strftime('%Y-%m').values)
                    ds_o = ds_o.assign_coords(time=ds_o.time.dt.strftime('%Y-%m').values)
                elif mode == "climatology":
                    ds_m = ds_m.mean(dim='time')
                    ds_o = ds_o.mean(dim='time')

                r, sd, rmsd = get_stats(ds_o, ds_m)
                
                if r is not None:
                    style = model_styles[m_name]
                    add_label = m_name if (row_idx == 0 and col_idx == 0) else ""
                    ax.plot(np.arccos(r), sd, style['m'], color=style['c'], markersize=14, 
                            label=add_label, markeredgecolor='black', 
                            markeredgewidth=1.5, alpha=1.0, zorder=15)
                    all_stats.append([var, mode.capitalize(), m_name, round(r, 4), round(sd, 4), round(rmsd, 4)])
                    
            except Exception: 
                pass

        # E. LABELING: TICK MARK & CUSTOM LABELING
        tick_vals = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
        ax.set_xticks(np.arccos(tick_vals))
        ax.set_xticklabels([str(t) for t in tick_vals])
        ax.tick_params(axis='x', labelsize=14, pad=12) 
        
        # PERUBAHAN: Correlation Coefficient HANYA di Kanan Atas (Baris 0, Kolom 1)
        if row_idx == 0 and col_idx == 1:
            ax.text(np.deg2rad(45), 1.55, "Correlation Coefficient", rotation=-45, 
                    ha='center', va='bottom', fontweight='bold', fontsize=17)

        sd_ticks = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4]
        ax.set_yticks(sd_ticks)
        ax.set_yticklabels([str(s) for s in sd_ticks])
        ax.tick_params(axis='y', labelsize=14)
        ax.set_rlabel_position(90) 
        
        # PERUBAHAN: Normalized Stdev HANYA di Kiri Bawah (Baris 1, Kolom 0)
        if row_idx == 1 and col_idx == 0:
            # FIX: Pake transAxes biar mutlak di tengah-bawah bingkai panel
            ax.text(0.5, -0.08, "Normalized Standard Deviation", transform=ax.transAxes, 
                    ha='center', va='top', fontweight='bold', fontsize=17)
        
        ax.set_ylim(0, 1.5)
        
        # F. LABEL PANEL (A, B, C, D)
        panel_letter = panel_labels[row_idx][col_idx]
        ax.text(-0.02, 1.0, panel_letter, transform=ax.transAxes, 
                fontsize=28, va='top', ha='right', color='black')
        
        ax.grid(True, linestyle='--', alpha=0.7, color='dimgray', linewidth=1.5)

# --- 3. LAYOUT & LEGEND ---
# MATIKAN tight_layout agar tidak bentrok dengan subplots_adjust
# plt.tight_layout() # <-- DIBUANG

# GUNAKAN wspace=0.1 untuk memaksa panel kiri dan kanan berdempetan presisi
plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.20, wspace=-0.35, hspace=0.15)

handles, labels = axes[0, 0].get_legend_handles_labels()

# Legend 4 kolom, 2 baris (ncol=4) dipusatkan di bawah figur
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.15), 
           fontsize=15, markerscale=1, labelspacing=1, borderpad=1, 
           frameon=True, edgecolor='black', ncol=4, columnspacing=3.0)

# Save Output
plt.savefig(os.path.join(output_path, "Taylor_Final_Multipanel_Compact.png"), dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(output_path, "Taylor_Final_Multipanel_Compact.svg"), format='svg', bbox_inches='tight')
plt.show()

# Save Tabulasi CSV
df_stats = pd.DataFrame(all_stats, columns=['Variable', 'Mode', 'Model', 'R', 'SD_norm', 'RMSD_norm'])
df_stats.to_csv(os.path.join(output_path, "Statistics_Taylor_Final_All.csv"), index=False)

print("\n--- DONE! Layout presisi, legend 4x2 di bawah, stdev persis di tengah panel C. ---")

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os
import warnings

warnings.filterwarnings("ignore")

# --- 1. KONFIGURASI DIREKTORI ---
LOCAL_RUN = True  
if LOCAL_RUN:
    output_path = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\1-seleksi_model\output"
else:
    output_path = os.path.join("data", "1-seleksi_model", "output")

tss_data_path = os.path.join(output_path, "tss_data")

models = ["CanESM5", "EC-Earth3", "IPSL-CM6A-LR", "MPI-ESM1-2-HR", 
          "MPI-ESM1-2-LR", "NorESM2-LM", "NorESM2-MM"]
variables = ["sfcWind", "rsds"] # Kolom 0: sfcWind, Kolom 1: rsds

lon_min, lon_max = 94.5, 141.5
lat_min, lat_max = -11.5, 7

# --- 2. MAIN VISUALIZATION LOOP ---
for mode in ["monthly"]:
    print(f"\nMembuat Visualisasi TSS Mode: {mode.capitalize()}")
    
    # Grid 7 Baris x 2 Kolom. Tinggi fig (20) disesuaikan biar Indonesia ga gepeng
    fig, axes = plt.subplots(7, 2, figsize=(14, 20), 
                             subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Atur jarak supaya bener-bener rapat (seamless)
    plt.subplots_adjust(wspace=-0.075, hspace=0.02, left=0.05, right=0.95, top=0.92, bottom=0.08)
    
    im = None 

    for row_idx, m_name in enumerate(models):
        for col_idx, var in enumerate(variables):
            ax = axes[row_idx, col_idx]
            
            nc_file = os.path.join(tss_data_path, f"tss_{mode}_{var}_{m_name}.nc")
            
            # Cek apakah file hasil komputasi ada
            if not os.path.exists(nc_file):
                ax.set_visible(False)
                continue
                
            try:
                # Load instant
                tss_map = xr.open_dataarray(nc_file)
                
                ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
                
                im = tss_map.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Spectral', 
                                  vmin=0, vmax=0.7, add_colorbar=False, add_labels=False)
                
                ax.coastlines(resolution='10m', color='black', linewidth=0.5)
                ax.gridlines(draw_labels=False, linestyle=':', alpha=0.2, color='gray')
                
                # Tulis nama model HANYA di kolom kiri (sfcWind) untuk menghindari repetisi
                if col_idx == 0:
                    ax.text(-0.02, 0.5, m_name, transform=ax.transAxes, fontsize=14, 
                            fontweight='bold', ha='right', va='center', rotation=90)
                
                # Tulis judul variabel HANYA di baris paling atas
                if row_idx == 0:
                    title_text = "sfcWind" if var == "sfcWind" else "rsds"
                    ax.set_title(title_text, fontweight='bold', fontsize=18, pad=15)
                    
            except Exception as e:
                print(f"Error plot {m_name} {var}: {e}")
                ax.set_visible(False)

    if im:
        # Colorbar Global di Bawah (Memanjang mencakup 2 kolom)
        # Parameter [left, bottom, width, height]
        cbar_ax = fig.add_axes([0.07, 0.055, 0.855, 0.015]) 
        cb = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', extend='max')
        cb.set_label('TSS', fontsize=16, fontweight='bold', labelpad=10)
        cb.ax.tick_params(labelsize=14)

    # Super Title untuk membedakan output Daily dan Monthly
    #plt.suptitle(f"Taylor Skill Score (TSS) - {mode.capitalize()} (1991-2020)", 
                 #fontsize=22, fontweight='bold', y=0.98)
    
    # Save High-Res
    plt.savefig(os.path.join(output_path, f"TSS_Seamless_{mode.capitalize()}_MultiVar.png"), 
                dpi=300, bbox_inches='tight')
    # Save SVG buat naskah Semhas
    plt.savefig(os.path.join(output_path, f"TSS_Seamless_{mode.capitalize()}_MultiVar.svg"), 
                format='svg', bbox_inches='tight')
    
    plt.show()

print("\n--- VISUALISASI SELESAI MAS! Plot 7x2 siap masuk dokumen Semhas. ---")